# 04. 외부 공공데이터 교차검증 (Step 4) ⭐ 차별화 핵심

법무부 출입국·외국인정책본부 「등록외국인 지역별 현황(2026년 6월말 기준)」의
시군구×체류자격(대분류) 통계를 카드데이터 세그먼트 분류와 결합하여,
**"생활밀착형 세그먼트는 실제로 취업계열 체류자격 비중이 유의하게 높은가?"** 를 검증한다.

- 출처: moj.go.kr 통계월보 게시판 (bbs/immigration/227/608715), 공공누리 4유형
- 기준시점: 2026-06-30 — 카드데이터 마지막 집계월(202606)과 시점 차이 없음

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from external_data import (
    load_moj_visa_by_region, compute_visa_group_shares,
    merge_segment_with_visa, run_visa_validation, VISA_GROUPS,
)
from region_mapping import build_join_report

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)


## 4-1. 행정구역명 매핑 검증 (조인 실패 건수 로그)

In [2]:
card = pd.read_csv('../data/raw/ABP_CONTEST_DATA.csv', encoding='utf-8-sig', dtype=str)
card_pairs = set(zip(card['SIDO_NM'], card['CCG_NM']))

moj_raw = load_moj_visa_by_region()
moj_pairs = set(zip(moj_raw['시도'], moj_raw['시군구']))

report = build_join_report(card_pairs, moj_pairs)
print(f"조인 대상 지역 수: {report['total']}")
print(f"매칭 성공: {report['matched']}개 ({(1-report['fail_rate'])*100:.1f}%)")
print(f"매칭 실패: {report['unmatched']}개 ({report['fail_rate']*100:.1f}%) -> {report['unmatched_list']}")

if report['fail_rate'] > 0.20:
    print("⚠️ 경고: 매칭 실패율이 20%를 초과합니다 (§9-2 확인 필요 상황) — 결과 신뢰성 재검토 필요.")
else:
    print("[확인] 매칭 실패율이 낮아(20% 미만) 자동 진행 가능 (§9-2 기준 충족).")


조인 대상 지역 수: 255
매칭 성공: 255개 (100.0%)
매칭 실패: 0개 (0.0%) -> []
[확인] 매칭 실패율이 낮아(20% 미만) 자동 진행 가능 (§9-2 기준 충족).


## 4-2. 체류자격 재범주화

30여 개 체류자격 대분류 코드를 3개 분석용 그룹으로 재범주화한다.

| 그룹 | 포함 자격 |
|---|---|
| 취업계열 | E7(특정활동), E8(계절근로), E9(비전문취업), E10(선원취업), H2(방문취업) |
| 유학계열 | D2(유학), D4(일반연수) |
| 정주계열 | F5(영주), F6(결혼이민) — **F4(재외동포)는 이 표에 없음** (거소신고 별도 관리, 한계점으로 명시) |


In [3]:
shares = compute_visa_group_shares(moj_raw)
print(f"법무부 자료 지역 수: {len(shares)}")
shares[['시도', '시군구', '총합계', '취업계열_비중', '유학계열_비중', '정주계열_비중']].describe()


법무부 자료 지역 수: 256


,총합계,취업계열_비중,유학계열_비중,정주계열_비중
count,256.000000,256.000000,256.000000,256.000000
mean,6394.226562,41.652203,16.216916,21.710586
std,6779.969970,27.515439,18.719994,13.717440
min,1.000000,1.293610,0.000000,0.000000
25%,1787.500000,14.434357,1.533566,11.436931
50%,4141.500000,41.408589,8.753368,17.354559
75%,8109.750000,65.720596,24.375509,27.074291
max,40479.000000,100.000000,76.983753,67.306248


## 4-3. 세그먼트 분류 결과와 결합

In [4]:
segment_df = pd.read_csv('../results/segment_classification.csv')
merged = merge_segment_with_visa(segment_df, shares)
print(f"결합 결과: {merged.shape[0]}개 지역 (조인 실패 0건 — merge_segment_with_visa 내부 assert 통과)")
merged[['SIDO_NM', 'CCG_NM', 'segment', '취업계열_비중', '유학계열_비중', '정주계열_비중']].head()


결합 결과: 255개 지역 (조인 실패 0건 — merge_segment_with_visa 내부 assert 통과)


,SIDO_NM,CCG_NM,segment,취업계열_비중,유학계열_비중,정주계열_비중
0,강원특별자치도,강릉시,프리미엄외식형,23.614567,35.421850,14.860891
1,강원특별자치도,고성군,프리미엄외식형,25.339186,63.367917,3.551476
2,강원특별자치도,동해시,로컬미식형,48.454636,2.891326,26.121635
3,강원특별자치도,삼척시,로컬미식형,39.188371,34.827377,12.053301
4,강원특별자치도,속초시,로컬미식형,38.453608,28.711340,16.288660


## 4-4. ⭐ 핵심 검증: 생활밀착형 vs 나머지 — 취업계열 비중 차이

Mann-Whitney U 검정 사용 (세그먼트별 표본 크기가 다르고, 비중(%) 데이터가 정규분포를
보장하지 않아 비모수 검정을 채택). 참고용으로 3개 세그먼트 전체 비교(Kruskal-Wallis)도 함께 수행.

In [5]:
validation = run_visa_validation(merged, seg_a_label='생활밀착형')
validation


,체류자격그룹,비교,생활밀착형_평균,생활밀착형_중앙값,나머지_평균,나머지_중앙값,검정방법,통계량,p_value,유의(p<0.05)
0,취업계열,생활밀착형 vs 나머지,59.112097,59.507231,33.188990,23.557283,Mann-Whitney U,10850.000000,4.073117e-12,True
1,유학계열,생활밀착형 vs 나머지,7.613672,4.616626,20.315074,12.310628,Mann-Whitney U,4649.000000,1.229072e-05,True
2,정주계열,생활밀착형 vs 나머지,14.329377,12.578616,25.271440,20.068910,Mann-Whitney U,3622.000000,4.230897e-10,True
3,취업계열,3개 세그먼트 전체,NaN,NaN,NaN,NaN,Kruskal-Wallis,79.088774,6.700248e-18,True
4,유학계열,3개 세그먼트 전체,NaN,NaN,NaN,NaN,Kruskal-Wallis,72.619399,1.701758e-16,True
5,정주계열,3개 세그먼트 전체,NaN,NaN,NaN,NaN,Kruskal-Wallis,47.361222,5.195666e-11,True


In [6]:
row = validation[(validation['체류자격그룹'] == '취업계열') & (validation['비교'] == '생활밀착형 vs 나머지')].iloc[0]
print(f"생활밀착형 취업계열 비중 평균: {row['생활밀착형_평균']:.1f}%  (나머지: {row['나머지_평균']:.1f}%)")
print(f"Mann-Whitney U = {row['통계량']:.1f}, p-value = {row['p_value']:.2e}")
if row['유의(p<0.05)']:
    print("[결론] 가설 채택: 생활밀착형 세그먼트는 통계적으로 유의하게(p<0.05) 취업계열 체류자격 비중이 높다.")
    print("       -> 카드데이터만으로는 '추정'이었던 거주형 소비 패턴이 법무부 데이터로 '검증된 사실'로 격상됨.")
else:
    print("[결론] 가설 기각: 통계적으로 유의한 차이가 확인되지 않았다. (반증도 유효한 분석 결과이므로 그대로 보고)")


생활밀착형 취업계열 비중 평균: 59.1%  (나머지: 33.2%)
Mann-Whitney U = 10850.0, p-value = 4.07e-12
[결론] 가설 채택: 생활밀착형 세그먼트는 통계적으로 유의하게(p<0.05) 취업계열 체류자격 비중이 높다.
       -> 카드데이터만으로는 '추정'이었던 거주형 소비 패턴이 법무부 데이터로 '검증된 사실'로 격상됨.


## 4-4b. ⭐ 강건성 점검: 법무부 저표본(등록외국인 총합계) 지역을 제외해도 결론이 유지되는가

법무부 등록외국인 총합계(지역별 등록외국인 수)는 지역 간 편차가 극단적으로 크다
(실측 최소 1명 ~ 최대 40,479명). 총합계가 아주 작은 지역은 체류자격 비중(%)이
개인 1~2명 단위로도 크게 흔들린다 — 예를 들어 등록외국인이 단 1명뿐인 지역은
그 1명의 자격이 곧바로 100%/0%가 되어 버린다. 카드데이터 쪽에는 이런 저표본
지역을 걸러내는 `low_sample` 플래그가 이미 있었지만, 법무부 데이터 쪽에는
동일한 점검이 빠져 있었다.

같은 기준(하위 10분위)으로 법무부 총합계 저표본 지역을 정의하고, 이 지역들을
제외한 뒤 §4-4의 핵심 검정(생활밀착형 vs 나머지, 취업/유학/정주계열 비중 차이)을
다시 수행해 원래 결론이 소수 지역의 불안정한 비중값 때문에 생긴 착시가 아님을 확인한다.

In [7]:
low_sample_regions = shares.loc[shares['moj_low_sample'], ['시도', '시군구', '총합계']].sort_values('총합계')
threshold = shares.attrs.get('moj_low_sample_threshold')
print(f"법무부 저표본(moj_low_sample) 임계치(총합계 하위 10분위): {threshold:.0f}명")
print(f"저표본 지역 수: {shares['moj_low_sample'].sum()}개 / {len(shares)}개")
print("\n저표본 지역 중 총합계가 가장 작은 5곳 (극단 사례 확인용):")
print(low_sample_regions.head(5).to_string(index=False))

법무부 저표본(moj_low_sample) 임계치(총합계 하위 10분위): 1015명
저표본 지역 수: 26개 / 256개

저표본 지역 중 총합계가 가장 작은 5곳 (극단 사례 확인용):
  시도 시군구   총합계
전라북도 고창군   1.0
경상북도 울릉군 152.0
전라남도 구례군 233.0
 경기도 과천시 266.0
충청남도 계룡시 280.0


In [8]:
from external_data import run_visa_validation_robustness

robustness = run_visa_validation_robustness(merged, seg_a_label='생활밀착형')
robustness

,체류자격그룹,제외지역수,잔여지역수,생활밀착형_평균(저표본제외),나머지_평균(저표본제외),p_value(저표본제외),유의(p<0.05)
0,취업계열,25,230,58.315472,29.362270,1.213869e-14,True
1,유학계열,25,230,7.903884,22.878600,5.774859e-08,True
2,정주계열,25,230,14.548508,25.880996,2.451748e-09,True


In [9]:
row_orig = validation[(validation['체류자격그룹'] == '취업계열') & (validation['비교'] == '생활밀착형 vs 나머지')].iloc[0]
row_robust = robustness[robustness['체류자격그룹'] == '취업계열'].iloc[0]

print("[비교] 취업계열 비중 차이 — 원본 vs 저표본 제외 후")
print(f"  원본        : {row_orig['생활밀착형_평균']:.1f}% vs {row_orig['나머지_평균']:.1f}%, p={row_orig['p_value']:.3e}")
print(f"  저표본 제외 : {row_robust['생활밀착형_평균(저표본제외)']:.1f}% vs {row_robust['나머지_평균(저표본제외)']:.1f}%, "
      f"p={row_robust['p_value(저표본제외)']:.3e} ({row_robust['제외지역수']}개 지역 제외, 잔여 {row_robust['잔여지역수']}개)")

if robustness['유의(p<0.05)'].all():
    print("\n[결론] 저표본 지역을 제외해도 3개 체류자격그룹 모두 유의성이 그대로 유지된다 "
          "(오히려 p-value가 더 작아지는 경우도 있음). §4-4의 핵심 결론은 소수 지역의 "
          "불안정한 비중값에 의존한 결과가 아니라 전체 데이터에 걸쳐 안정적으로 나타나는 패턴이다.")
else:
    print("\n[결론] 저표본 지역 제외 시 일부 그룹에서 유의성이 사라진다 — "
          "원본 결과의 강건성에 대한 재검토가 필요하다.")

[비교] 취업계열 비중 차이 — 원본 vs 저표본 제외 후
  원본        : 59.1% vs 33.2%, p=4.073e-12
  저표본 제외 : 58.3% vs 29.4%, p=1.214e-14 (25개 지역 제외, 잔여 230개)

[결론] 저표본 지역을 제외해도 3개 체류자격그룹 모두 유의성이 그대로 유지된다 (오히려 p-value가 더 작아지는 경우도 있음). §4-4의 핵심 결론은 소수 지역의 불안정한 비중값에 의존한 결과가 아니라 전체 데이터에 걸쳐 안정적으로 나타나는 패턴이다.


In [10]:
from pathlib import Path
robustness.to_csv(Path('../results/segment_visa_validation_robustness.csv'), index=False, encoding='utf-8-sig')
print('[OK] 저장: ../results/segment_visa_validation_robustness.csv')

[OK] 저장: ../results/segment_visa_validation_robustness.csv


## 4-5. 세그먼트별 체류자격 구성비 시각적 요약

In [11]:
summary = merged.groupby('segment')[['취업계열_비중', '유학계열_비중', '정주계열_비중']].mean().round(1)
summary = summary.reindex(['생활밀착형', '로컬미식형', '프리미엄외식형'])
summary


,취업계열_비중,유학계열_비중,정주계열_비중
segment,,,
생활밀착형,59.1,7.6,14.3
로컬미식형,39.4,12.9,27.7
프리미엄외식형,15.3,41.6,18.2


## 4-6. 산출물 저장: `results/segment_visa_validation.csv`

In [12]:
from pathlib import Path
out_path = Path('../results/segment_visa_validation.csv')
validation.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"[OK] 저장: {out_path}")


[OK] 저장: ../results/segment_visa_validation.csv


## 한계점 (보고서 반영 필수)

1. **F-4(재외동포) 미포함**: 법무부 「등록외국인」 통계는 외국인등록 제도 기준이며, 재외동포(F-4)는
   국내거소신고로 별도 관리되어 이 표에 집계되지 않는다. 정주계열 비중이 실제보다 과소 추정되었을 수 있다.
2. **집계 단위 차이**: 카드데이터는 "소비 발생 지역" 기준이고 법무부 데이터는 "등록 거주지" 기준이다.
   관광객처럼 거주지와 소비지가 다른 경우 두 데이터의 지역 개념이 완전히 같지 않다.
3. **시점**: 두 데이터 모두 2026년 6월 기준으로 시점 차이는 없으나, 법무부 통계는 월말 스냅샷(저량,stock)이고
   카드데이터는 6개월 누적 흐름(유량, flow)이라는 성격 차이가 있다.
4. **저표본 지역(해결됨, §4-4b 참조)**: 법무부 등록외국인 총합계는 지역별 편차가 극단적으로 커(최소 1명
   ~최대 40,479명) 비중값이 불안정한 지역이 섞여 있었다. §4-4b에서 하위 10분위(25개 지역) 제외 후
   재검정한 결과 핵심 결론이 유지(오히려 강화)됨을 확인해, 이 한계가 결론에 실질적 영향을 주지
   않음을 검증했다.

다음 단계(`05_analysis.ipynb`)에서 세그먼트별 심층 비교와 이상치 케이스 스터디를 진행한다.